# 04 — Temperature and rental demand
 **RQ4:** At which observed temperatures are average rental counts highest?

 The bin with the highest descriptive mean is **not** an experimentally established *optimal temperature*. Season, hour and system growth may influence these averages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

hour=pd.read_csv('../data/processed/bike_hour_clean.csv')
Path('../output/figures').mkdir(parents=True,exist_ok=True)
hour['temp_c']=hour['temp']*41
hour['temp_bin']=pd.cut(hour['temp_c'],bins=np.arange(0,44,2),right=False)
summary=(hour.groupby('temp_bin',observed=True)
         .agg(n=('cnt','size'),registered=('registered','mean'),
              casual=('casual','mean')).reset_index())
summary['temp_mid']=summary['temp_bin'].apply(lambda x:x.mid).astype(float)
used=summary.loc[summary['n']>=40].copy()

fig,axes=plt.subplots(1,2,figsize=(12,4.9),sharey=False)
for ax,key,color,title in zip(axes,['registered','casual'],
                              ['#2563EB','#E69F00'],['Registered users','Casual users']):
    ax.plot(used['temp_mid'],used[key],color=color,marker='o',linewidth=2.3)
    highest=used.loc[used[key].idxmax()]
    ax.scatter([highest['temp_mid']],[highest[key]],s=55,color=color,zorder=4)
    ax.annotate(f'Highest bin: {highest["temp_mid"]:.0f}°C',
                xy=(highest['temp_mid'],highest[key]),xytext=(-95,-38),
                textcoords='offset points',fontsize=9,
                arrowprops={'arrowstyle':'->','color':'#555555'})
    ax.set(title=title,xlabel='Temperature (°C)',ylabel='Average rentals per observed hour')
    ax.set_ylim(bottom=0);ax.grid(axis='y',alpha=.18)
    print(f'{title}: highest mean among displayed bins at {highest["temp_mid"]:.0f}°C '
          f'(n={int(highest["n"])} hours in bin)')
fig.suptitle('Rentals tend to rise with temperature within the observed range',
             fontsize=13,fontweight='bold',y=1.03)
fig.text(.01,-.04,'2°C bins; bins with <40 hours excluded. Panel y-scales differ. '
         'Descriptive, not causal. UCI / Capital Bikeshare, 2011–2012.',fontsize=9,color='#555555')
fig.tight_layout()
fig.savefig('../output/figures/RQ4_temperature.png',dpi=200,bbox_inches='tight')
plt.show()
display(used[['temp_bin','n','registered','casual']].round(1))

### Interpretation and limitations
 `temp` is normalized in the raw data and converted to °C as specified by the dataset Readme. Relative demand cannot be inferred from just the plotted means without considering hourly and seasonal composition. The 40-hour threshold is a display choice and should be disclosed.